# Notebook 01 — ETL Pipeline — Student Version

## Goal

Build a **versioned, privacy-aware ML feature dataset** from the QBC12 Airbnb PostgreSQL database.

Final output:

- one row per `listing_id`
- one fixed `cutoff_date`
- features built only from data available before/on the cutoff
- target built from future calendar availability
- no raw PII columns in the final ML dataset

The next notebook will use this output for MLflow experiments. If this ETL is messy, the ML notebook will be garbage.

## What you must submit from this notebook

By the end, your notebook must save these files under `data/features/`:

```text
listing_availability_features_<version>.csv
listing_availability_features_<version>.parquet
listing_availability_features_<version>_metadata.json
listing_availability_features_<version>_validation_report.json
pii_audit_<version>.csv
```

The notebook must also show:

1. database connection check,
2. table/column inspection,
3. PII audit,
4. cutoff-date logic,
5. feature construction,
6. label construction,
7. validation checks.

## 0. Imports

These libraries are enough for the ETL notebook.

Install missing packages with:

```bash
pip install pandas numpy sqlalchemy psycopg2-binary pyarrow
```

In [1]:
import os
import json
import re
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

## 1. Configuration

These values define the dataset version and the time windows.

- `PAST_WINDOW_DAYS`: how much history is used for features.
- `FUTURE_WINDOW_DAYS`: how much future data is used for the target.
- `HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD`: the rule for the positive class.

If you change any of these, change `DATASET_VERSION`.

In [2]:
# -----------------------------
# ETL Configuration
# -----------------------------
DATASET_VERSION = "v1_student"

ENTITY_COLUMN = "listing_id"

PAST_WINDOW_DAYS = 90
FUTURE_WINDOW_DAYS = 30
HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD = 0.30

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
FEATURE_DIR = DATA_DIR / "features"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURE_DIR:", FEATURE_DIR)

PROJECT_ROOT: g:\work\MLops\mlops-bootcamp\hw11_etl-pipeline
FEATURE_DIR: g:\work\MLops\mlops-bootcamp\hw11_etl-pipeline\data\features


## 2. Database connection

Use your assigned student database user.

The QBC12 database is:

host: 185.50.38.163

port: 32112

database: qbc12_airbnb

Important:

- Keep `sslmode=disable`.
- Do not commit real passwords to Git.

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
# -----------------------------
# Database Connection
# -----------------------------

# Clear old environment variables that may point to the wrong database.
# for key in ["PGHOST", "PGPORT", "PGDATABASE", "PGUSER", "PGPASSWORD"]:
#     os.environ.pop(key, None)

DB_HOST = os.getenv("PGHOST", "")
DB_PORT = int(os.getenv("PGPORT", "32112"))
DB_NAME = os.getenv("PGDATABASE", "")
DB_USER = os.getenv("PGUSER", "")
DB_PASSWORD = os.getenv("PGPASSWORD", "")


if DB_USER == "student_your_username" or DB_PASSWORD == "student_your_password":
    raise ValueError("Replace DB_USER and DB_PASSWORD with your assigned database credentials.")

print("Connecting to:")
print("HOST:", DB_HOST)
print("PORT:", DB_PORT)
print("DB:", DB_NAME)
print("USER:", DB_USER)

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    query={"sslmode": "disable"},
)

engine = create_engine(db_url)

def read_sql(query: str, params: dict | None = None) -> pd.DataFrame:
    """Run a SQL query through SQLAlchemy and return a Pandas DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, params=params)

with engine.connect() as conn:
    connection_check = conn.execute(
        text("""
        SELECT
            current_database() AS database,
            current_user AS user_name,
            inet_server_addr() AS server_ip,
            inet_server_port() AS server_port,
            now() AS checked_at;
        """)
    ).mappings().first()

dict(connection_check)

Connecting to:
HOST: 185.50.38.163
PORT: 32112
DB: qbc12_airbnb
USER: student_hamed_hamzeh


{'database': 'qbc12_airbnb',
 'user_name': 'student_hamed_hamzeh',
 'server_ip': '172.19.0.2',
 'server_port': 5432,
 'checked_at': datetime.datetime(2026, 6, 5, 8, 8, 32, 620900, tzinfo=datetime.timezone.utc)}

## 3. Inspect the available data

Before writing ETL, inspect the database.

You should confirm:

- which tables exist,
- which columns exist,
- how many rows each table has,
- whether important fields are missing.

In [6]:
tables_df = read_sql("""
SELECT
    table_schema,
    table_name,
    table_type
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY table_schema, table_name;
""")

tables_df

,table_schema,table_name,table_type
0,core,calendar_day,BASE TABLE
1,core,host,BASE TABLE
2,core,listing,BASE TABLE
3,core,neighbourhood,BASE TABLE
4,core,review,BASE TABLE


In [ ]:
columns_df = read_sql("""
SELECT
    table_schema,
    table_name,
    ordinal_position,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'core'
ORDER BY table_schema, table_name, ordinal_position;
""")

columns_df

In [7]:
row_counts_df = read_sql("""
SELECT 'core.calendar_day' AS table_name, COUNT(*) AS row_count FROM core.calendar_day
UNION ALL
SELECT 'core.host' AS table_name, COUNT(*) AS row_count FROM core.host
UNION ALL
SELECT 'core.listing' AS table_name, COUNT(*) AS row_count FROM core.listing
UNION ALL
SELECT 'core.neighbourhood' AS table_name, COUNT(*) AS row_count FROM core.neighbourhood
UNION ALL
SELECT 'core.review' AS table_name, COUNT(*) AS row_count FROM core.review
ORDER BY table_name;
""")

row_counts_df

,table_name,row_count
0,core.calendar_day,3825200
1,core.host,9201
2,core.listing,10480
3,core.neighbourhood,22
4,core.review,501084


## 4. Data quality audit

This step decides which columns are safe and useful.

You must check at least:

1. calendar date range,
2. review date range,
3. whether `calendar_day.price` and `adjusted_price` are usable,
4. whether recent review windows are meaningful.

Do not include columns that are all-null or nearly useless.

In [8]:
calendar_quality_df = read_sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE price IS NULL) AS null_price,
    COUNT(*) FILTER (WHERE adjusted_price IS NULL) AS null_adjusted_price,
    COUNT(*) FILTER (WHERE available IS NULL) AS null_available,
    MIN(date) AS min_calendar_date,
    MAX(date) AS max_calendar_date
FROM core.calendar_day;
""")

review_quality_df = read_sql("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE comment_len IS NULL) AS null_comment_len,
    MIN(review_date) AS min_review_date,
    MAX(review_date) AS max_review_date
FROM core.review;
""")

display(calendar_quality_df)
display(review_quality_df)

,n_rows,null_price,null_adjusted_price,null_available,min_calendar_date,max_calendar_date
0,3825200,3825200,3825200,0,2025-09-11,2026-09-10


,n_rows,null_comment_len,min_review_date,max_review_date
0,501084,0,2010-08-22,2025-09-11


In [10]:
# Inspect small samples.
# Keep LIMIT small. Do not pull full raw calendar/review tables into Pandas.

for table_name in ["listing", "host", "neighbourhood", "review", "calendar_day"]:
    print(f"\n===== core.{table_name} =====")
    display(read_sql(f"SELECT * FROM core.{table_name} LIMIT 5;"))


===== core.listing =====


,listing_id,host_id,neighbourhood_id,room_type,property_type,accommodates,bedrooms,beds,bathrooms_text,listing_price,minimum_nights,maximum_nights,instant_bookable,license
0,27886,97647,2,Private room,Private room in houseboat,2,1.0,1.0,1.5 baths,132.0,3,356,False,0363 974D 4986 7411 88D8
1,28871,124245,2,Private room,Private room in rental unit,2,1.0,1.0,1 shared bath,89.0,2,730,False,0363 607B EA74 0BD8 2F6F
2,29051,124245,1,Private room,Private room in condo,2,1.0,1.0,1 shared bath,61.0,2,730,False,0363 607B EA74 0BD8 2F6F
3,44391,194779,1,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5 baths,NaN,3,730,False,0363 E76E F06A C1DD 172C
4,48373,220434,8,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5 baths,NaN,3,1125,False,0363 4A2B A6AD 0196 F684



===== core.host =====


,host_id,host_pseudo_id,is_superhost
0,27837566,12a252de05fbf2f7ba9f57fa3baa099acd17e2a9c7efc7...,False
1,12840373,d9f7e79668b99a5cb7963bfb2430d8b6b960a0ec4e82bb...,False
2,226859324,cc90d30412e286c0525c7914d031807c8c00e017fe1915...,True
3,20204265,755d79bf4be51df2d85792123200e6641db8589551965e...,True
4,47981094,b40c45156e81696746b6eb51ef86aeccff7c6d7cd5eac2...,False



===== core.neighbourhood =====


,neighbourhood_id,name
0,1,Centrum-Oost
1,2,Centrum-West
2,3,Oostelijk Havengebied - Indische Buurt
3,4,Westerpark
4,5,Slotervaart



===== core.review =====


,review_id,listing_id,review_date,reviewer_id,reviewer_pseudo_id,comment_len
0,531281017,18062995,2019-09-17,73925523,2408ff6b678c0bdc30857dc6631d8762b57159ef9741c3...,392
1,531788922,18062995,2019-09-18,82572265,fb2e9dd6e8225de8231d388c9b9cf92b9c88f7e0823389...,13
2,532695264,18062995,2019-09-20,100595030,2503cafbac72a2a0a211282e2ed6e97c60058769ab3cf8...,467
3,537292491,18062995,2019-09-28,282542584,2612adccbea98416289b033feb314e33a3947244104fcf...,78
4,540324583,18062995,2019-10-03,116115856,7f6c33a5339d594d253c1aa0c9d5a2ae714321907af14c...,220



===== core.calendar_day =====


,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,857771032141263073,2026-06-20,False,None,None,2,45
1,857771032141263073,2026-06-21,False,None,None,2,45
2,857771032141263073,2026-06-22,False,None,None,2,45
3,857771032141263073,2026-06-23,False,None,None,2,45
4,857771032141263073,2026-06-24,False,None,None,2,45


## 5. Choose the cutoff date

The cutoff separates features from the label.

Rules:

- Historical features use dates `history_start_date <= date <= cutoff_date`.
- The label uses dates `cutoff_date < date <= label_end_date`.
- The cutoff must have enough past calendar data and enough future calendar data.

For this homework, use a 90-day feature window and a 30-day label window.

In [11]:
range_df = read_sql("""
SELECT
    (SELECT MIN(date) FROM core.calendar_day) AS calendar_min_date,
    (SELECT MAX(date) FROM core.calendar_day) AS calendar_max_date,
    (SELECT MIN(review_date) FROM core.review) AS review_min_date,
    (SELECT MAX(review_date) FROM core.review) AS review_max_date;
""")

range_df

,calendar_min_date,calendar_max_date,review_min_date,review_max_date
0,2025-09-11,2026-09-10,2010-08-22,2025-09-11


In [ ]:
calendar_min_date = pd.to_datetime(range_df.loc[0, "calendar_min_date"]).date()
calendar_max_date = pd.to_datetime(range_df.loc[0, "calendar_max_date"]).date()

# TODO:
# 1. Compute earliest valid cutoff using calendar_min_date and PAST_WINDOW_DAYS.
# 2. Compute latest valid cutoff using calendar_max_date and FUTURE_WINDOW_DAYS.
# 3. Choose a valid cutoff_date.
# 4. Compute history_start_date and label_end_date.
#
# Hint:
# earliest valid cutoff = calendar_min_date + PAST_WINDOW_DAYS
# latest valid cutoff = calendar_max_date - FUTURE_WINDOW_DAYS

earliest_cutoff_allowed_by_calendar = None  # TODO
latest_cutoff_allowed_by_calendar = None    # TODO

cutoff_date = None          # TODO
history_start_date = None   # TODO
label_end_date = None       # TODO

# TODO: print all dates and assert that the windows are valid.
raise NotImplementedError("TODO: implement cutoff-date logic.")

## 6. PII audit

Raw identifiers can be needed for joins, but they must not become model features.

Your final ML feature table must not contain:

- `host_id`
- `host_pseudo_id`
- `review_id`
- `reviewer_id`
- `reviewer_pseudo_id`
- `license`
- raw text fields that may contain sensitive information

`listing_id` may stay as an entity key, but it must be excluded from model inputs later.

In [ ]:
# TODO: complete the PII audit table.
# Add rows for all sensitive or identity-linking columns you find relevant.

pii_audit = pd.DataFrame([
    {
        "table": "listing",
        "column": "listing_id",
        "pii_type": "entity identifier",
        "decision": "keep as entity key only",
        "reason": "needed to define one row per listing; not a model input"
    },
    # TODO: add host_id
    # TODO: add host_pseudo_id
    # TODO: add license
    # TODO: add review_id
    # TODO: add reviewer_id
    # TODO: add reviewer_pseudo_id
])

pii_audit

## 7. Extract static tables

`listing`, `host`, and `neighbourhood` are small enough to load directly.

Do not load full `review` or `calendar_day` into Pandas. Those must be aggregated in SQL later.

In [ ]:
# TODO: write SQL to load the required listing columns.
listing_df = read_sql("""
SELECT
    -- TODO: select only needed columns
FROM core.listing;
""")

# TODO: write SQL to load host columns.
host_df = read_sql("""
SELECT
    -- TODO: select only needed columns
FROM core.host;
""")

# TODO: write SQL to load neighbourhood columns.
neighbourhood_df = read_sql("""
SELECT
    -- TODO: select needed columns and rename name to neighbourhood_name
FROM core.neighbourhood;
""")

print("listing:", listing_df.shape)
print("host:", host_df.shape)
print("neighbourhood:", neighbourhood_df.shape)

## 8. Clean static fields

Convert database values into ML-friendly columns.

Required work:

- convert booleans to boolean dtype,
- convert numeric listing columns to numeric dtype,
- parse `bathrooms_text` into a numeric `bathrooms` feature.

In [ ]:
# TODO: normalize boolean columns.
# Example target columns:
# - listing_df["instant_bookable"]
# - host_df["is_superhost"]

# TODO: normalize numeric listing columns.
numeric_cols_listing = [
    "accommodates",
    "bedrooms",
    "beds",
    "listing_price",
    "minimum_nights",
    "maximum_nights",
]

for col in numeric_cols_listing:
    # TODO
    pass


def parse_bathrooms(text_value):
    """
    Convert bathrooms_text into a number.

    Examples:
    - '1 bath' -> 1.0
    - '1.5 baths' -> 1.5
    - 'Half-bath' -> 0.5
    - missing/unrecognized -> NaN
    """
    # TODO: implement parser.
    raise NotImplementedError("TODO: implement parse_bathrooms")


listing_df["bathrooms"] = listing_df["bathrooms_text"].apply(parse_bathrooms)

listing_df[["bathrooms_text", "bathrooms"]].head(10)

## 9. Build static listing features

Join:

- `listing` → `host`
- `listing` → `neighbourhood`
- host-level aggregate `host_listing_count`

Final static features should be one row per `listing_id`.

Do not keep raw `host_id`, `host_pseudo_id`, `neighbourhood_id`, `license`, or `bathrooms_text` in the final static feature table.

In [ ]:
# TODO: create host_listing_features with one row per host_id.
host_listing_features = None

# TODO: merge listing, host, host_listing_features, and neighbourhood.
base_listing_features = None

# TODO: choose privacy-safe static feature columns.
static_feature_cols = [
    "listing_id",
    # TODO
]

static_features = base_listing_features[static_feature_cols].copy()

assert static_features["listing_id"].duplicated().sum() == 0

print(static_features.shape)
static_features.head()

## 10. Build review features in SQL

Do not load raw `core.review` into Pandas.

Build one row per listing in SQL.

Required output columns:

- `listing_id`
- `total_reviews_before_cutoff`
- `unique_reviewers_before_cutoff`
- `avg_comment_len_before_cutoff`
- `max_comment_len_before_cutoff`
- `days_since_last_review`

Use only reviews where `review_date <= cutoff_date`.

In [ ]:
# TODO: write SQL aggregation over core.review.
# Use CAST(:cutoff_date AS date) when using the cutoff inside SQL.

review_features = read_sql(
    """
    SELECT
        listing_id
        -- TODO: aggregate review features
    FROM core.review
    WHERE review_date <= CAST(:cutoff_date AS date)
    GROUP BY listing_id;
    """,
    params={"cutoff_date": cutoff_date},
)

# TODO: convert feature columns to numeric where needed.

assert review_features["listing_id"].duplicated().sum() == 0

print(review_features.shape)
review_features.head()

## 11. Build calendar history features in SQL

Do not load raw `core.calendar_day` into Pandas.

Build historical availability features using:

- 90-day history window,
- 30-day recent history window.

Do not include calendar price features unless your audit proves they are usable.

In [ ]:
# TODO: write SQL aggregation over core.calendar_day.
# Use history_start_date and cutoff_date.
#
# Required output examples:
# - available_days_last_90d
# - available_rate_last_90d
# - avg_minimum_nights_calendar_last_90d
# - avg_maximum_nights_calendar_last_90d
# - available_days_last_30d
# - available_rate_last_30d
# - avg_minimum_nights_calendar_last_30d
# - avg_maximum_nights_calendar_last_30d

calendar_features_all = read_sql(
    """
    SELECT
        listing_id
        -- TODO: aggregate calendar history features
    FROM core.calendar_day
    WHERE date >= CAST(:history_start_date AS date)
      AND date <= CAST(:cutoff_date AS date)
    GROUP BY listing_id;
    """,
    params={
        "history_start_date": history_start_date,
        "cutoff_date": cutoff_date,
    },
)

# TODO: convert numeric columns.

assert calendar_features_all["listing_id"].duplicated().sum() == 0

print(calendar_features_all.shape)
calendar_features_all.head()

## 12. Build the target label

The label is built from future calendar availability.

Positive class:

```text
high_demand_proxy = 1 if future_available_rate_30d <= 0.30
```

This is not confirmed booking demand. It is a low-availability proxy.

In [ ]:
# TODO: write SQL to build one label row per listing.
# Use only dates after cutoff_date and up to label_end_date.

label_df = read_sql(
    """
    SELECT
        listing_id
        -- TODO: future_calendar_days_observed_30d
        -- TODO: future_available_days_30d
        -- TODO: future_available_rate_30d
        -- TODO: high_demand_proxy
    FROM core.calendar_day
    WHERE date > CAST(:cutoff_date AS date)
      AND date <= CAST(:label_end_date AS date)
    GROUP BY listing_id;
    """,
    params={
        "cutoff_date": cutoff_date,
        "label_end_date": label_end_date,
        "threshold": HIGH_DEMAND_AVAILABLE_RATE_THRESHOLD,
    },
)

# TODO: convert numeric columns and make high_demand_proxy integer.

assert label_df["listing_id"].duplicated().sum() == 0

print(label_df.shape)
label_df.head()

In [ ]:
# Check label balance.

label_distribution = (
    label_df["high_demand_proxy"]
    .value_counts(dropna=False)
    .rename_axis("high_demand_proxy")
    .reset_index(name="count")
)

label_distribution["percentage"] = (
    label_distribution["count"] / label_distribution["count"].sum()
).round(4)

label_distribution

## 13. Join feature groups and label

Join all feature groups into one ML-ready table.

The final granularity must be:

```text
one row = one listing_id at one cutoff_date
```

Use an inner join with `label_df`, because rows without a target cannot be used for supervised learning.

In [ ]:
# TODO: join static_features, review_features, calendar_features_all, and label_df.
feature_df = None

# TODO: add cutoff_date and dataset_version columns.
# TODO: fill missing review count features with zero.
# TODO: handle missing days_since_last_review for listings with no reviews.

assert feature_df["listing_id"].duplicated().sum() == 0

print(feature_df.shape)
feature_df.head()

## 14. Drop unusable columns

Before saving, remove bad feature columns.

Drop columns that are:

- more than 95% missing,
- constant across all rows,

but protect target/audit columns.

In [ ]:
protected_columns = {
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
}

HIGH_MISSING_DROP_THRESHOLD = 0.95

# TODO: compute missing rates.
# TODO: find high-missing columns outside protected columns.
# TODO: find constant columns outside protected columns.
# TODO: drop them from feature_df.

columns_to_drop = []  # TODO

print("Columns to drop:", columns_to_drop)

feature_df = feature_df.drop(columns=columns_to_drop)

print("New shape:", feature_df.shape)

## 15. Validate the final dataset

The validation step is mandatory.

Check:

1. no duplicate `listing_id + cutoff_date`,
2. target exists and is binary,
3. no missing target values,
4. no forbidden PII columns,
5. no future leakage columns in model inputs.

In [ ]:
duplicate_count = feature_df.duplicated(subset=["listing_id", "cutoff_date"]).sum()
missing_target_count = feature_df["high_demand_proxy"].isna().sum()
unique_target_values = sorted(feature_df["high_demand_proxy"].dropna().unique().tolist())

forbidden_columns = {
    "host_id",
    "host_pseudo_id",
    "reviewer_id",
    "reviewer_pseudo_id",
    "review_id",
    "license",
    "bathrooms_text",
}

present_forbidden_columns = sorted(forbidden_columns.intersection(feature_df.columns))

label_only_columns = [
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

model_input_columns = [
    col for col in feature_df.columns
    if col not in label_only_columns
    and col not in ["listing_id", "cutoff_date", "dataset_version"]
]

future_leakage_columns = [
    col for col in model_input_columns
    if col.startswith("future_")
]

# TODO: add asserts for each validation rule.
# for example:
# assert duplicate_count == 0

print("duplicate_count:", duplicate_count)
print("missing_target_count:", missing_target_count)
print("unique_target_values:", unique_target_values)
print("present_forbidden_columns:", present_forbidden_columns)
print("future_leakage_columns:", future_leakage_columns)
print("model_input_column_count:", len(model_input_columns))

In [ ]:
missing_report = (
    feature_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing_report.columns = ["column", "missing_rate"]

calendar_coverage_summary = (
    feature_df[["future_calendar_days_observed_30d"]]
    .describe()
    .reset_index()
)

display(missing_report.head(30))
display(label_distribution)
display(calendar_coverage_summary)

## 16. Save versioned outputs

Save:

- feature dataset,
- metadata,
- validation report,
- PII audit.

The MLflow notebook must read this output instead of querying raw database tables again.

In [ ]:
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"
validation_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_validation_report.json"
pii_audit_path = FEATURE_DIR / f"pii_audit_{DATASET_VERSION}.csv"

feature_df.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path)

try:
    feature_df.to_parquet(parquet_path, index=False)
    print("Saved Parquet:", parquet_path)
except ImportError:
    print("Parquet not saved because pyarrow/fastparquet is not installed.")
    print("Install pyarrow with: pip install pyarrow")

# TODO: build metadata dictionary.
metadata = {
    "dataset_version": DATASET_VERSION,
    "entity_column": ENTITY_COLUMN,
    # TODO: add cutoff, windows, source tables, target definition, exclusion rules
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# TODO: build validation_report dictionary.
validation_report = {
    "duplicate_listing_cutoff_rows": int(duplicate_count),
    "missing_target_count": int(missing_target_count),
    "target_values": unique_target_values,
    "present_forbidden_columns": present_forbidden_columns,
    "future_leakage_columns_in_model_inputs": future_leakage_columns,
    # TODO: add missing_report, label_distribution, calendar_coverage_summary
}

with open(validation_path, "w", encoding="utf-8") as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)

pii_audit.to_csv(pii_audit_path, index=False)

print("Saved metadata:", metadata_path)
print("Saved validation report:", validation_path)
print("Saved PII audit:", pii_audit_path)

## 17. Final preview

Use this final cell to confirm the output shape and columns.

Before moving to Notebook 2, make sure:

- target column exists,
- model input columns do not include future columns,
- no forbidden PII columns are present,
- saved files exist in `data/features/`.

In [ ]:
print("Final shape:", feature_df.shape)

display(feature_df.head())

feature_df.info()